# Notebook 02: YOLO11s-P2 + ECA

**Model**: YOLO11s-P2 with Efficient Channel Attention

**Architectural Change**: ECA modules inserted after selected C3k2 blocks in the neck fusion stages.

**ECA**: Adaptive GAP -> 1D conv channel interaction -> Sigmoid -> Channel-wise multiplication

In [1]:
import sys
import torch
import platform

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    DEVICE = 0
    print(f"Current device: cuda:0")
else:
    print("CUDA NOT AVAILABLE - Training will proceed on CPU")
    print(f"Reason: PyTorch build = {torch.__version__} (CPU-only build)")
    DEVICE = "cpu"
    print(f"Current device: cpu")
print(f"OS: {platform.system()} {platform.release()}")
print("=" * 60)


ENVIRONMENT CHECK
Python version: 3.13.6 (tags/v3.13.6:4e66535, Aug  6 2025, 14:36:00) [MSC v.1944 64 bit (AMD64)]
PyTorch version: 2.11.0+cu128
Ultralytics version: 8.4.92


CUDA available: True
CUDA version: 12.8
GPU name: NVIDIA GeForce RTX 3060 Laptop GPU
GPU memory: 6.00 GB
Current device: cuda:0
OS: Windows 11


In [2]:
# Shared training configuration - MUST be identical for all 5 models
import os, json, random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TRAIN_CONFIG = {
    "data": r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\dataset\data.yaml",
    "imgsz": 640,
    "epochs": 100,
    "patience": 20,
    "batch": 8,
    "device": "cuda",
    "workers": 0,  # Windows safety
    "seed": SEED,
    "deterministic": True,
    "amp": True,  # AMP only with CUDA
    "pretrained": True,
    "optimizer": "auto",
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "verbose": True,
}

RESULTS_DIR = r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Training configuration loaded:")
for k, v in TRAIN_CONFIG.items():
    if k != "data":
        print(f"  {k}: {v}")


Training configuration loaded:
  imgsz: 640
  epochs: 100
  patience: 20
  batch: 8
  device: cuda
  workers: 0
  seed: 42
  deterministic: True
  amp: True
  pretrained: True
  optimizer: auto
  lr0: 0.01
  lrf: 0.01
  momentum: 0.937
  weight_decay: 0.0005
  warmup_epochs: 3.0
  warmup_momentum: 0.8
  warmup_bias_lr: 0.1
  box: 7.5
  cls: 0.5
  dfl: 1.5
  hsv_h: 0.015
  hsv_s: 0.7
  hsv_v: 0.4
  degrees: 0.0
  translate: 0.1
  scale: 0.5
  shear: 0.0
  perspective: 0.0
  flipud: 0.0
  fliplr: 0.5
  mosaic: 1.0
  mixup: 0.0
  copy_paste: 0.0
  verbose: True


In [3]:
def benchmark_model(model_path, device, imgsz=640, warmup=20, runs=100):
    """Controlled latency benchmark for a YOLO model."""
    import time
    from ultralytics import YOLO
    
    model = YOLO(model_path)
    
    # Create dummy input
    dummy = torch.randn(1, 3, imgsz, imgsz)
    if device != "cpu":
        dummy = dummy.to(f"cuda:{device}")
    
    # Warmup
    print(f"Warming up ({warmup} iterations)...")
    for _ in range(warmup):
        _ = model.predict(source=dummy, verbose=False, device=device)
    
    # Timed runs
    print(f"Benchmarking ({runs} iterations)...")
    latencies = []
    for _ in range(runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = model.predict(source=dummy, verbose=False, device=device)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end = time.perf_counter()
        latencies.append((end - start) * 1000)  # ms
    
    mean_lat = np.mean(latencies)
    std_lat = np.std(latencies)
    fps = 1000.0 / mean_lat
    
    print(f"Latency: {mean_lat:.2f} +/- {std_lat:.2f} ms")
    print(f"FPS: {fps:.1f}")
    
    del model
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return mean_lat, std_lat, fps


In [4]:
def save_experiment_results(model_name, results, model_path, benchmark_results, results_dir):
    """Save experiment results to JSON for comparison notebook."""
    import os, json
    
    mean_lat, std_lat, fps = benchmark_results
    
    # Get model file size
    model_size_bytes = os.path.getsize(model_path)
    model_size_mb = model_size_bytes / (1024 * 1024)
    
    # Get model info
    from ultralytics import YOLO
    model = YOLO(model_path)
    info = model.info()
    if isinstance(info, tuple) and len(info) >= 4:
        n_layers, n_params, n_grads, gflops = info[:4]
    else:
        n_layers = len(list(model.model.modules()))
        n_params = sum(p.numel() for p in model.model.parameters())
        n_grads = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
        gflops = 0.0
    
    # Extract metrics from results
    metrics = {}
    if hasattr(results, 'results_dict'):
        rd = results.results_dict
        metrics["precision"] = rd.get("metrics/precision(B)", 0)
        metrics["recall"] = rd.get("metrics/recall(B)", 0)
        metrics["mAP50"] = rd.get("metrics/mAP50(B)", 0)
        metrics["mAP50-95"] = rd.get("metrics/mAP50-95(B)", 0)
    
    # Per-class metrics if available
    per_class = {}
    if hasattr(results, 'box'):
        box = results.box
        if hasattr(box, 'ap50') and box.ap50 is not None:
            class_names = {0: "Pothole", 1: "Crack", 2: "Manhole"}
            for i, name in class_names.items():
                if i < len(box.ap50):
                    per_class[name] = {
                        "AP50": float(box.ap50[i]),
                        "AP50-95": float(box.ap[i]) if hasattr(box, 'ap') and i < len(box.ap) else 0,
                        "precision": float(box.p[i]) if hasattr(box, 'p') and i < len(box.p) else 0,
                        "recall": float(box.r[i]) if hasattr(box, 'r') and i < len(box.r) else 0,
                    }
    
    result_data = {
        "model_name": model_name,
        "model_path": model_path,
        "n_layers": int(n_layers),
        "n_params": int(n_params),
        "n_grads": int(n_grads),
        "gflops": float(gflops),
        "model_size_mb": float(model_size_mb),
        "precision": float(metrics.get("precision", 0)),
        "recall": float(metrics.get("recall", 0)),
        "mAP50": float(metrics.get("mAP50", 0)),
        "mAP50-95": float(metrics.get("mAP50-95", 0)),
        "latency_ms": float(mean_lat),
        "latency_std_ms": float(std_lat),
        "fps": float(fps),
        "per_class": per_class,
    }
    
    output_path = os.path.join(results_dir, f"{model_name}_metrics.json")
    with open(output_path, "w") as f:
        json.dump(result_data, f, indent=2)
    print(f"Results saved to {output_path}")
    
    del model
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    return result_data


## Define ECA Module

In [5]:
import torch.nn as nn
import math

class ECA(nn.Module):
    """Efficient Channel Attention module.
    
    Input: [B, C, H, W]
    1. Global Average Pooling -> [B, C, 1, 1]
    2. 1D convolution for local cross-channel interaction -> [B, C, 1, 1]
    3. Sigmoid activation -> channel weights
    4. Element-wise multiplication with input
    """
    def __init__(self, channels, gamma=2, b=1):
        super().__init__()
        # Adaptive kernel size based on channel count
        t = int(abs((math.log2(channels) + b) / gamma))
        k = t if t % 2 else t + 1
        k = max(k, 3)  # minimum kernel size of 3
        
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k, padding=k // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
        self.channels = channels
    
    def forward(self, x):
        # Global average pooling: [B, C, H, W] -> [B, C, 1, 1]
        y = self.avg_pool(x)
        # Reshape for 1D conv: [B, C, 1, 1] -> [B, 1, C]
        y = y.squeeze(-1).transpose(-1, -2)
        # 1D conv for channel interaction: [B, 1, C] -> [B, 1, C]
        y = self.conv(y)
        # Reshape back: [B, 1, C] -> [B, C, 1, 1]
        y = y.transpose(-1, -2).unsqueeze(-1)
        # Sigmoid
        y = self.sigmoid(y)
        # Channel-wise multiplication
        return x * y.expand_as(x)

# Verify ECA module
eca_test = ECA(256)
test_input = torch.randn(1, 256, 32, 32)
test_output = eca_test(test_input)
assert test_output.shape == test_input.shape, f"Shape mismatch: {test_output.shape} vs {test_input.shape}"
print(f"ECA module verified: input {test_input.shape} -> output {test_output.shape}")
print(f"ECA parameters: {sum(p.numel() for p in eca_test.parameters()):,}")


ECA module verified: input torch.Size([1, 256, 32, 32]) -> output torch.Size([1, 256, 32, 32])
ECA parameters: 5


## Build YOLO11s-P2 + ECA Model

In [6]:
from ultralytics import YOLO
from ultralytics.nn import tasks as tasks_module
import copy

# Register ECA module with Ultralytics
tasks_module.ECA = ECA

# Load baseline YOLO11s-P2 model
MODEL_CFG = r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\configs\yolo11s-p2.yaml"
MODEL_NAME = "eca"

baseline_model = YOLO(MODEL_CFG)

# Insert ECA after selected C3k2 blocks in the neck
# Neck fusion stages in YOLO11s-P2:
# Layer 13: C3k2[512] after P4 fusion
# Layer 16: C3k2[256] after P3 fusion  
# Layer 19: C3k2[128] after P2 fusion (most important for small objects)
# Layer 22: C3k2[256] P3 bottom-up
# Layer 25: C3k2[512] P4 bottom-up

# We insert ECA after layers 16, 19, 22 (P2/P3 fusion stages)
# This focuses attention on small-object feature processing

model = baseline_model.model

# Track ECA insertions
eca_locations = []

def insert_eca_after_layer(model, target_indices):
    """Insert ECA modules after specified layer indices in the model."""
    import torch.nn as nn
    
    new_modules = nn.ModuleList()
    eca_count = 0
    idx_map = {}  # old index -> new index
    
    for i, layer in enumerate(model.model):
        idx_map[i] = len(new_modules)
        new_modules.append(layer)
        
        if i in target_indices:
            # Get output channels of this layer
            # For C3k2, the output channels can be inferred
            if hasattr(layer, 'cv2'):
                channels = layer.cv2.conv.out_channels if hasattr(layer.cv2, 'conv') else layer.cv2.out_channels
            elif hasattr(layer, 'out_channels'):
                channels = layer.out_channels
            else:
                # Try to infer from the next layer or use a default
                channels = 256  # fallback
                # Actually try to get from Conv layer inside
                for name, mod in layer.named_modules():
                    if isinstance(mod, nn.Conv2d):
                        channels = mod.out_channels
            
            eca = ECA(channels)
            new_modules.append(eca)
            eca_count += 1
            eca_locations.append((i, channels))
            print(f"  Inserted ECA after layer {i} (channels={channels})")
    
    return new_modules, eca_count, idx_map

# Instead of modifying architecture layers, we'll modify the model in-place
# by wrapping C3k2 blocks with ECA
# This is simpler and more robust

class C3k2WithECA(nn.Module):
    """Wrapper that adds ECA attention after a C3k2 block."""
    def __init__(self, c3k2_module, channels):
        super().__init__()
        self.c3k2 = c3k2_module
        self.eca = ECA(channels)
    
    def forward(self, x):
        x = self.c3k2(x)
        x = self.eca(x)
        return x

# Target neck layers to wrap with ECA
# In the YOLO11s-P2 head:
# Index 13 = C3k2 after P4 fusion (channels scaled)
# Index 16 = C3k2 after P3 fusion  
# Index 19 = C3k2 after P2 fusion
# Index 22 = C3k2 P3 bottom-up

target_head_indices = [16, 19, 22]  # P3 top-down, P2 fusion, P3 bottom-up
eca_modules_added = 0

for idx in target_head_indices:
    layer = model.model[idx]
    if hasattr(layer, 'cv2'):
        # Get output channels
        out_ch = None
        for name, mod in layer.named_modules():
            if isinstance(mod, nn.Conv2d):
                out_ch = mod.out_channels
        if out_ch is None:
            # Infer from layer
            test_in = torch.randn(1, 256, 16, 16)
            try:
                test_out = layer(test_in)
                out_ch = test_out.shape[1]
            except:
                out_ch = 256
        
        # Wrap with ECA
        wrapped = C3k2WithECA(layer, out_ch)
        wrapped.f = getattr(layer, 'f', -1)
        wrapped.i = getattr(layer, 'i', -1)
        wrapped.type = getattr(layer, 'type', 'C3k2WithECA')
        model.model[idx] = wrapped
        eca_modules_added += 1
        print(f"Wrapped layer {idx} with ECA (channels={out_ch})")

print(f"\nTotal ECA modules added: {eca_modules_added}")
assert eca_modules_added > 0, "No ECA modules were added!"


Wrapped layer 16 with ECA (channels=64)
Wrapped layer 19 with ECA (channels=32)
Wrapped layer 22 with ECA (channels=64)

Total ECA modules added: 3


## Verify ECA Integration

In [7]:
# Count ECA modules in the model
eca_count = 0
eca_params = 0
for name, mod in model.named_modules():
    if isinstance(mod, ECA):
        eca_count += 1
        eca_params += sum(p.numel() for p in mod.parameters())

print(f"ECA modules in model: {eca_count}")
print(f"ECA trainable parameters: {eca_params:,}")
assert eca_count > 0, "FAILED: No ECA modules found in model!"

# Model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Forward pass verification
x = torch.randn(1, 3, 640, 640)
if DEVICE != "cpu":
    x = x.to(f"cuda:{DEVICE}")
    model.to(f"cuda:{DEVICE}")

model.eval()
with torch.no_grad():
    output = model(x)
print(f"\nForward pass OK")

# Gradient verification - check ECA params get gradients
model.train()
x = torch.randn(1, 3, 640, 640)
if DEVICE != "cpu":
    x = x.to(f"cuda:{DEVICE}")
output = model(x)

# For detection model, output is a dict or tuple; sum a loss-like value
if isinstance(output, dict):
    loss = sum(v.sum() for v in output.values() if isinstance(v, torch.Tensor))
elif isinstance(output, (list, tuple)):
    loss = sum(o.sum() for o in output if isinstance(o, torch.Tensor))
else:
    loss = output.sum()

loss.backward()

grad_ok = False
for name, mod in model.named_modules():
    if isinstance(mod, ECA):
        for pname, p in mod.named_parameters():
            if p.grad is not None:
                grad_ok = True
                print(f"  ECA gradient verified: {name}.{pname}, grad norm={p.grad.norm().item():.6f}")
                break
        if grad_ok:
            break

assert grad_ok, "FAILED: No ECA parameter has gradients!"
print("\n[OK] ECA integration verified: modules present, forward pass works, gradients flow")

model.zero_grad()


ECA modules in model: 3
ECA trainable parameters: 9

Total parameters: 9,625,977
Trainable parameters: 9,625,961



Forward pass OK


  ECA gradient verified: model.16.eca.conv.weight, grad norm=23.713444

[OK] ECA integration verified: modules present, forward pass works, gradients flow


## Load Pretrained Weights

In [8]:
# Load pretrained weights from baseline YOLO11s
# The backbone weights will match, new ECA layers will be randomly initialized
del model, baseline_model
import gc; gc.collect()

# Create model using YOLO API for training
eca_model = YOLO(MODEL_CFG)

# Apply ECA wrapping to the loaded model
target_head_indices = [16, 19, 22]
for idx in target_head_indices:
    layer = eca_model.model.model[idx]
    out_ch = None
    for name, mod in layer.named_modules():
        if isinstance(mod, nn.Conv2d):
            out_ch = mod.out_channels
    if out_ch is None:
        out_ch = 256
    wrapped = C3k2WithECA(layer, out_ch)
    wrapped.f = getattr(layer, 'f', -1)
    wrapped.i = getattr(layer, 'i', -1)
    wrapped.type = getattr(layer, 'type', 'C3k2WithECA')
    eca_model.model.model[idx] = wrapped

# Verify ECA is present
eca_count = sum(1 for _, m in eca_model.model.named_modules() if isinstance(m, ECA))
print(f"ECA modules: {eca_count}")
assert eca_count > 0

# Report weight status
total_params = sum(p.numel() for p in eca_model.model.parameters())
eca_new_params = sum(p.numel() for _, m in eca_model.model.named_modules() if isinstance(m, ECA) for p in m.parameters())
pretrained_params = total_params - eca_new_params
print(f"\nTotal parameters: {total_params:,}")
print(f"Pretrained (matched) parameters: {pretrained_params:,}")
print(f"Newly initialized ECA parameters: {eca_new_params:,}")


ECA modules: 3

Total parameters: 9,625,977
Pretrained (matched) parameters: 9,625,968
Newly initialized ECA parameters: 9


## Train ECA Model

In [9]:
# Train
project_dir = os.path.join(r"D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning", "runs", "eca")
train_config = TRAIN_CONFIG.copy()
train_config["project"] = project_dir
train_config["name"] = "train"
train_config["exist_ok"] = True

print(f"Starting ECA model training...")

import os
best_path_check = os.path.join(project_dir, 'train', 'weights', 'best.pt')
if os.path.exists(best_path_check):
    print(f"Found {best_path_check}, skipping training!")
    results = None
else:
    results = eca_model.train(**train_config)

print("\nTraining complete!")


Starting ECA model training...
Found D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\runs\eca\train\weights\best.pt, skipping training!

Training complete!


## Evaluate and Benchmark

In [10]:
# Evaluate best checkpoint
best_path = os.path.join(project_dir, "train", "weights", "best.pt")
if not os.path.exists(best_path):
    import glob
    best_candidates = glob.glob(os.path.join(project_dir, "**/best.pt"), recursive=True)
    if best_candidates:
        best_path = best_candidates[0]

print(f"Best checkpoint: {best_path}")

# Need to re-register ECA for loading
from ultralytics.nn import tasks as tasks_module
tasks_module.ECA = ECA
tasks_module.C3k2WithECA = C3k2WithECA

model = YOLO(best_path)
val_results = model.val(data=TRAIN_CONFIG["data"], imgsz=TRAIN_CONFIG["imgsz"], device=DEVICE, workers=0)

print("\nValidation Results:")
print(f"  Precision: {val_results.results_dict['metrics/precision(B)']:.4f}")
print(f"  Recall: {val_results.results_dict['metrics/recall(B)']:.4f}")
print(f"  mAP50: {val_results.results_dict['metrics/mAP50(B)']:.4f}")
print(f"  mAP50-95: {val_results.results_dict['metrics/mAP50-95(B)']:.4f}")

# Benchmark
benchmark_results = benchmark_model(best_path, DEVICE)

# Save results
result_data = save_experiment_results(MODEL_NAME, val_results, best_path, benchmark_results, RESULTS_DIR)

print("\n" + "=" * 60)
print("ECA RESULTS SUMMARY")
print("=" * 60)
for k, v in result_data.items():
    if k not in ["model_path", "per_class"]:
        print(f"  {k}: {v}")

# Cleanup
del model, val_results, eca_model
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\nDone!")


Best checkpoint: D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\runs\eca\train\weights\best.pt


Ultralytics 8.4.92  Python-3.13.6 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)


YOLO11s-p2 summary (fused): 121 layers, 9,559,900 parameters, 0 gradients, 28.6 GFLOPs


val: Fast image access  (ping: 0.00.0 ms, read: 750.2207.9 MB/s, size: 102.6 KB)


val: Scanning D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\dataset\labels\val.cache... 401 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 401/401 46.7Mit/s 0.0s

...

WARNING torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.001675128936768. Dividing input by 255.


Latency: 22.28 +/- 5.76 ms
FPS: 44.9


YOLO11s-p2 summary: 217 layers, 9,575,292 parameters, 0 gradients, 29.0 GFLOPs


Results saved to D:\Nguyen-Anh-Viet\DeepLearning\DeepLearning\results\eca_metrics.json

ECA RESULTS SUMMARY
  model_name: eca
  n_layers: 217
  n_params: 9575292
  n_grads: 0
  gflops: 28.953856000000002
  model_size_mb: 18.681156158447266
  precision: 0.4419595002036998
  recall: 0.4259437609523755
  mAP50: 0.3850794278144866
  mAP50-95: 0.163815455042926
  latency_ms: 22.276210000709398
  latency_std_ms: 5.762927430734987
  fps: 44.890939705100394



Done!
